# Phase 1 — Knee preprocessing (self-contained)

Run on Kaggle (CPU notebook) with **only the competition dataset attached**.
No repo upload needed — this notebook writes the `knee` package itself
(payload embedded verbatim from `src/knee/preprocess.py`).

Output: `/kaggle/working/shards/` (part-*.npz + meta.csv + series_selection.csv)
→ **Save Version → Save & Run All → New Dataset** named `knee-shards`.

Pipeline: 3 series/study by plane/fluid priority, 8 slices/series sorted by
ImagePositionPatient[2], 224x224 uint8, per-volume percentile normalization.
~4.7 GB total, ~1.5-2h with 4 workers.

In [ ]:
import os

os.makedirs("/kaggle/working/knee", exist_ok=True)

with open("/kaggle/working/knee/__init__.py", "w") as f:
    f.write('"""knee: RSNA knee abnormality challenge package."""\n\n__version__ = "0.2.0"\n')

with open("/kaggle/working/knee/constants.py", "w", encoding="utf-8") as f:
    f.write(
        '\"\"\"Shared constants for the knee package.\"\"\"\n\n'
        'from typing import List\n\n'
        'STUDY_ID = "StudyInstanceUID"\n\n'
        'LABELS: List[str] = [\n'
        '    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",\n'
        '    "Medial OA", "Lateral OA", "PF OA",\n'
        '    "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture",\n'
        ']\n\n'
        'NUM_LABELS = len(LABELS)  # 12\n'
    )
print("package skeleton written")


In [ ]:
# knee/preprocess.py source — embedded VERBATIM from repo src/knee/preprocess.py.
# Regenerate the notebook with notebooks/build_preprocess_ipynb.py; do not edit.
_PP_PAYLOAD = '"""Shared preprocessing logic for the knee challenge (single source of truth).\n\nUsed by:\n- notebooks/10-preprocess.py   (Kaggle one-pass: train corpus -> npz shards)\n- notebooks/40-infer.py        (test-time preprocessing, efficiency-tuned)\n- local synthetic tests\n\nConstants (from Phase 0 recon):\n- 3 series per study by priority: sag-fluid > cor-fluid > ax-fluid,\n  fallbacks sag-nonfluid > cor-nonfluid > ax-nonfluid  (100% coverage verified)\n- 8 slices per series, uniformly sampled, sorted by ImagePositionPatient[2]\n  (100% present in recon; InstanceNumber fallback)\n- 224x224 uint8 (percentile-normalized per volume: p1/p99.5 clip -> scale)\n- per-volume normalization handles the mixed uint16/int16, 256-1024 matrices\n"""\n\nimport os\nimport re\nfrom typing import Dict, List, Optional, Tuple\n\nimport numpy as np\n\ntry:\n    import pydicom\nexcept ImportError as e:  # pragma: no cover\n    raise ImportError("pydicom required") from e\n\nIMG_SIZE = 224\nNUM_SLICES = 8\nMAX_SERIES = 3\n\n_SERIES_PRIORITY = [\n    ("Sagittal", 1), ("Coronal", 1), ("Axial", 1),\n    ("Sagittal", 0), ("Coronal", 0), ("Axial", 0),\n]\n\n\ndef find_data_dir(base: str = "/kaggle/input") -> str:\n    """Locate the competition dir; handles nested /kaggle/input/competitions/<slug>/.\n\n    Requires the match to contain train_series/ (the DICOM corpus root) so a\n    repo/dataset copy of train.csv can never be mistaken for the competition\n    data. Bounded-depth walk; prunes train_series/test_series DICOM trees.\n    """\n    if not os.path.isdir(base):\n        raise FileNotFoundError(f"{base} does not exist — attach the competition dataset")\n    for root, dirs, files in os.walk(base):\n        if ("train.csv" in files or "test.csv" in files) and \\\n                os.path.isdir(os.path.join(root, "train_series")):\n            return root\n        depth = root[len(base):].count(os.sep) + (1 if root != base else 0)\n        if depth >= 3:\n            dirs[:] = []\n        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]\n    raise FileNotFoundError(\n        "competition data not found under " + base +\n        " (need a directory with train.csv AND train_series/)"\n    )\n\n\ndef select_series(study_series_rows, max_series: int = MAX_SERIES) -> List[str]:\n    """Pick up to max_series SeriesInstanceUIDs by plane/fluid priority.\n\n    `study_series_rows`: iterable of dicts/Series-like with keys\n    Anatomical_Plane, Fluid_Sensitive, SeriesInstanceUID.\n    Deterministic (sorted UIDs) so train/test selections match.\n    """\n    by_bucket: Dict[Tuple[str, int], List[str]] = {}\n    for row in study_series_rows:\n        plane = str(row["Anatomical_Plane"])\n        fluid = int(row["Fluid_Sensitive"])\n        by_bucket.setdefault((plane, fluid), []).append(str(row["SeriesInstanceUID"]))\n\n    chosen: List[str] = []\n    for bucket in _SERIES_PRIORITY:\n        cands = by_bucket.get(bucket, [])\n        if cands:\n            chosen.append(sorted(cands)[0])\n        if len(chosen) >= max_series:\n            break\n    # fill any remaining slots deterministically\n    for uid in sorted(u for cands in by_bucket.values() for u in cands):\n        if uid not in chosen and len(chosen) < max_series:\n            chosen.append(uid)\n    return chosen\n\n\ndef _slice_sort_key(path: str) -> Tuple[float, str]:\n    """Header-only read for position; ~1ms. Falls back to filename."""\n    try:\n        ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)\n        ipp = getattr(ds, "ImagePositionPatient", None)\n        if ipp is not None and len(ipp) >= 3:\n            return (float(ipp[2]), path)\n        num = getattr(ds, "InstanceNumber", None)\n        if num is not None:\n            return (float(num), path)\n    except Exception:\n        pass\n    return (0.0, path)\n\n\n_DCM_RE = re.compile(r"\\.dcm$", re.IGNORECASE)\n\n\ndef read_series(\n    series_dir: str,\n    num_slices: int = NUM_SLICES,\n    img_size: int = IMG_SIZE,\n    header_only_first: bool = False,\n) -> Optional[np.ndarray]:\n    """Read one DICOM series -> (num_slices, img_size, img_size) uint8.\n\n    Efficiency: header-scans ALL files (cheap) to sort + choose sample\n    positions, then pixel-decodes ONLY the sampled slices.\n\n    Returns None on unreadable/empty series (caller zero-fills).\n    """\n    try:\n        files = [os.path.join(series_dir, f) for f in os.listdir(series_dir)\n                 if _DCM_RE.search(f)]\n    except OSError:\n        return None\n    if not files:\n        return None\n\n    files.sort(key=_slice_sort_key)\n    n = len(files)\n    if n <= num_slices:\n        idx = list(range(n))\n    else:\n        idx = sorted(set(int(i) for i in np.linspace(0, n - 1, num_slices)))\n        # ensure exactly num_slices entries (pad by repeating neighbors if dedup shrank)\n        j = 0\n        while len(idx) < num_slices and j < n:\n            if j not in idx:\n                idx.append(j)\n            j += 1\n        idx = sorted(idx[:num_slices])\n\n    slices = []\n    for i in idx:\n        try:\n            ds = pydicom.dcmread(files[i], force=True)\n            img = ds.pixel_array.astype(np.float32)\n            slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)\n            inter = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)\n            if slope != 1.0 or inter != 0.0:\n                img = img * slope + inter\n            slices.append(img)\n        except Exception:\n            continue\n    if not slices:\n        return None\n\n    # pad to num_slices by repeating last slice\n    while len(slices) < num_slices:\n        slices.append(slices[-1])\n\n    vol = np.stack(slices)  # (S, H, W) float32\n\n    # per-volume percentile normalization -> uint8\n    finite = vol[np.isfinite(vol)]\n    if finite.size == 0:\n        return None\n    p_lo, p_hi = np.percentile(finite, [1.0, 99.5])\n    if p_hi <= p_lo:\n        p_hi = p_lo + 1e-6\n    vol = np.clip(vol, p_lo, p_hi)\n    vol = (vol - p_lo) / (p_hi - p_lo)\n\n    # resize\n    try:\n        import cv2\n        out = np.empty((vol.shape[0], img_size, img_size), dtype=np.uint8)\n        for k in range(vol.shape[0]):\n            r = cv2.resize(vol[k], (img_size, img_size), interpolation=cv2.INTER_AREA)\n            out[k] = np.clip(np.rint(r * 255.0), 0, 255).astype(np.uint8)\n        return out\n    except ImportError:\n        # PIL fallback (Kaggle always has cv2, local may not)\n        from PIL import Image\n        out = np.empty((vol.shape[0], img_size, img_size), dtype=np.uint8)\n        for k in range(vol.shape[0]):\n            im = Image.fromarray((vol[k] * 255).astype(np.uint8))\n            out[k] = np.array(im.resize((img_size, img_size), Image.BILINEAR))\n        return out\n\n\ndef study_tensor(\n    study_dir: str,\n    series_uids: List[str],\n    num_slices: int = NUM_SLICES,\n    img_size: int = IMG_SIZE,\n) -> np.ndarray:\n    """Stack selected series -> (MAX_SERIES, NUM_SLICES, H, W) uint8.\n\n    Missing/unreadable series are zero-filled so the shape is fixed.\n    """\n    out = np.zeros((MAX_SERIES, num_slices, img_size, img_size), dtype=np.uint8)\n    for k, suid in enumerate(series_uids[:MAX_SERIES]):\n        sdir = os.path.join(study_dir, suid)\n        t = read_series(sdir, num_slices, img_size)\n        if t is not None:\n            out[k] = t\n    return out\n\n\ndef process_study(args):\n    """Multiprocessing worker: (uid, series_uids, series_root) -> result tuple.\n\n    Kept at module scope (picklable). Returns\n    (uid, ok, tensor) — tensor is zero-filled (MAX_SERIES, NUM_SLICES, H, W)\n    on failure so downstream stacking never breaks.\n    """\n    uid, series_uids, series_root = args\n    try:\n        t = study_tensor(os.path.join(series_root, uid), series_uids)\n        return uid, True, t\n    except Exception:\n        return uid, False, np.zeros(\n            (MAX_SERIES, NUM_SLICES, IMG_SIZE, IMG_SIZE), np.uint8\n        )\n'

with open("/kaggle/working/knee/preprocess.py", "w", encoding="utf-8") as f:
    f.write(_PP_PAYLOAD)
import py_compile
py_compile.compile("/kaggle/working/knee/preprocess.py", doraise=True)
print("knee/preprocess.py written and syntax-checked:", len(_PP_PAYLOAD), "chars")


In [ ]:
# --- Phase 1 run (mirrors notebooks/10-preprocess.py) ---
import os
import sys
import time
from multiprocessing import Pool

import numpy as np
import pandas as pd

sys.path.insert(0, "/kaggle/working")

from knee.preprocess import (
    IMG_SIZE, MAX_SERIES, NUM_SLICES, find_data_dir, process_study, select_series,
)

SHARDS_PER_FILE = 2000
OUT_DIR = "/kaggle/working/shards"
N_WORKERS = 4

DATA_DIR = find_data_dir("/kaggle/input")
SERIES_ROOT = os.path.join(DATA_DIR, "train_series")
print("data dir:", DATA_DIR)

os.makedirs(OUT_DIR, exist_ok=True)

series_meta = pd.read_csv(os.path.join(DATA_DIR, "train_series.csv"))

t0 = time.time()
selections = {}
tasks = []
for uid, grp in series_meta.groupby("StudyInstanceUID"):
    uids = select_series(grp.to_dict("records"))
    selections[uid] = uids
    tasks.append((uid, uids))
tasks = [(uid, u, SERIES_ROOT) for uid, u in tasks]
print(f"series selection: {len(tasks)} studies in {time.time()-t0:.1f}s")

sel_rows = [
    {"StudyInstanceUID": uid, "S1": u[0] if len(u) > 0 else "",
     "S2": u[1] if len(u) > 1 else "", "S3": u[2] if len(u) > 2 else ""}
    for uid, u in selections.items()
]
pd.DataFrame(sel_rows).to_csv(os.path.join(OUT_DIR, "series_selection.csv"), index=False)


def _write_shard(buf, shard):
    uids = sorted(buf)
    arr = np.stack([buf[u] for u in uids])
    np.savez_compressed(
        os.path.join(OUT_DIR, f"part-{shard:04d}.npz"),
        uids=np.array(uids),
        data=arr,
    )


meta_rows = []
shard, in_shard, fails = 0, 0, 0
buf = {}

t0 = time.time()
with Pool(N_WORKERS) as pool:
    for uid, ok, tensor in pool.imap(process_study, tasks, chunksize=8):
        if not ok or tensor is None:
            fails += 1
            tensor = np.zeros((MAX_SERIES, NUM_SLICES, IMG_SIZE, IMG_SIZE), np.uint8)
        buf[uid] = tensor
        in_shard += 1

        if in_shard == SHARDS_PER_FILE:
            _write_shard(buf, shard)
            meta_rows.extend(
                {"StudyInstanceUID": u, "shard": shard, "offset": i}
                for i, u in enumerate(sorted(buf))
            )
            buf = {}
            shard += 1
            in_shard = 0
            elapsed = time.time() - t0
            done = shard * SHARDS_PER_FILE
            eta = elapsed / done * (len(tasks) - done) / 60
            print(f"  shard {shard} written ({done} studies, "
                  f"{elapsed/60:.1f} min, ETA {eta:.1f} min, fails={fails})")

if buf:
    _write_shard(buf, shard)
    meta_rows.extend(
        {"StudyInstanceUID": u, "shard": shard, "offset": i}
        for i, u in enumerate(sorted(buf))
    )

pd.DataFrame(meta_rows).to_csv(os.path.join(OUT_DIR, "meta.csv"), index=False)
print(f"done: {shard+1} shards, {len(tasks)} studies, {fails} failures, "
      f"{(time.time()-t0)/60:.1f} min")
print("output:", OUT_DIR)


## Next

1. Quick sanity below
2. **Save Version → Save & Run All → New Dataset** named `knee-shards`

In [ ]:
# sanity check
import os

import numpy as np

shards = sorted(f for f in os.listdir(OUT_DIR) if f.endswith(".npz"))
print("shards:", shards)
z = np.load(os.path.join(OUT_DIR, shards[0]))
print("shape:", z["data"].shape, "dtype:", z["data"].dtype)
print("study 0: max=", z["data"][0].max(), "mean=", z["data"][0].mean().round(1))
assert z["data"].shape[1:] == (3, 8, 224, 224)
assert z["data"].max() > 0
print("OK")
